# Explain the churn model to a retention manager

The retention manager: "Before my team acts on this list, I want to know what the model looks at, and why it put customer 5194 on it. And should we push everyone onto two-year contracts?" You explain the boosted churn model three ways, then write the answer for someone who does not read code.

## The starter

The cell fits `make_pipeline(tree_pre, HistGradientBoostingClassifier(random_state=42))` on the 5,634 training customers and prints its test ROC AUC. `tree_pre` imputes and one-hot encodes but does not scale, so every value you read is in months, dollars or categories.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)


def make_tree_pre(X):
    """Impute and one-hot encode, no scaling; output columns keep readable names (tenure, Contract_Two year)."""
    num_cols = list(X.select_dtypes(include="number").columns)
    cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ], verbose_feature_names_out=False).set_output(transform="pandas")


tree_pre = make_tree_pre(X)
model = make_pipeline(tree_pre, HistGradientBoostingClassifier(random_state=42)).fit(X_train, y_train)
print("test ROC AUC", round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3))

## Step 1 · Permutation importance on the test customers

`permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42)`. Put `importances_mean` and `importances_std` in a table sorted from largest to smallest.

## Step 2 · Which columns matter?

Which columns lose the most ROC AUC when shuffled? Which ones lose almost nothing?

## Step 3 · Partial dependence on `tenure`, with ICE lines

`PartialDependenceDisplay.from_estimator(model, X_test_f, ["tenure"], kind="both", subsample=50, random_state=42)`, where `X_test_f = X_test.astype({"tenure": "float64"})`. Read the average curve at 0 and at 72 months.

## Step 4 · Partial dependence on `Contract`

Use `categorical_features=["Contract"]`. Read the three values.

## Step 5 · One customer, explained with SHAP

Transform the test rows with the fitted preprocessing (`Xt = model[0].transform(X_test)`), build `explainer = shap.TreeExplainer(model[-1])` and `sv = explainer(Xt)`, then draw `shap.plots.waterfall(sv[i])` for the row of customer 5194 (`i = list(X_test.index).index(5194)`).

## Step 6 · The beeswarm

`shap.plots.beeswarm(sv)`. Which color sits on the right for `tenure`, and what does that mean?

## Step 7 · Four sentences for the manager

What does the model look at, why customer 5194, and should we push everyone onto two-year contracts? Use the top two columns from step 1 and the three largest pushes from step 5. Then say whether a model that predicts churn tells you what happens when you change a customer's contract.

## Bonus · Drop `TotalCharges`

## Check yourself